# Vidéo vers 3DGS sur Colab T4
Exécuter les cellules dans l'ordre. **Tout le calcul reste dans `/content`** : vidéo copiée une fois, frames, masques, SfM, entraînement et logs. Drive contient les entrées et les sauvegardes uniquement.

**CUDA ne suffit pas : Spirula exige Vulkan NVIDIA.** Le contrôle bloque avant l'extraction si le pilote n'est pas utilisable. Installer `libvulkan1` ne fournit pas le pilote NVIDIA.
Drive reçoit uniquement le ZIP COLMAP après préparation, puis le ZIP du `.splat` final et de son géoréférencement. Les checkpoints, masques, logs et autres artefacts restent locaux.
Les paramètres de la première cellule sont des variables Python directes : entrées principales, puis réglages rarement modifiés, puis valeurs figées. Pour changer de données, choisir un nouveau `RUN_NAME`. À paramètres identiques, « Tout exécuter » réutilise Conda, Spirula, les frames et les étapes déjà terminées.

Les logs sont affichés en direct et conservés localement. `/content` disparaît à la fin du runtime : sauvegarder après une phase ou après un échec. Aucune restauration automatique de workspace Drive, aucun effacement d'installation existante.


In [ ]:
from pathlib import Path
import os, subprocess, sys

# --- 1. Entrées principales ---
VIDEO_DRIVE = Path('/content/drive/MyDrive/gsplat/input/IMG_5017.MOV')
RUN_NAME = 'video_run'  # Nouveau nom si vidéo ou réglages changent.
NUM_FRAMES = 300  # Nombre de frames extraites de la vidéo.
MASKS = 'off'  # 'on' ou 'off' : appliqué à SFM_MASKS et TRAIN_MASKS.
TRAIN_NUM_ITER = 12000
TRAIN_SP_CAP_MAX = 500000  # Maximum de splats par bloc.
# --- 2. Réglages rarement modifiés ---
SPIRULA_ARCHIVE = Path('/content/drive/MyDrive/spirula-builds/spirula-studio-vulkan-20261114_203210.zip')
PIPELINE_RESUME = 'true'
REMOVE_LOCKS = 'false'  # true seulement après arrêt des anciens calculs.
GIT_BRANCH = 'review_copilot'
ACCEPT_SAM_LICENSE = True  # Licence SAM : https://github.com/facebookresearch/sam3/blob/main/LICENSE
NUM_CPUS = 2  # CPU parallélisables (SfM, masques SAM) ; réduire à 1 si VRAM insuffisante.
SFM_SP_PAIRS = 'sequential'  # prefilter, sequential, auto ou exhaustive.
SPLIT_NUM_BLOCKS = 1
DRIVE_BACKUPS = Path('/content/drive/MyDrive/colmap_to_gs_runs')
# --- 3. Valeurs figées ---
COLMAP_ARCHIVE = ''  # ZIP COLMAP désactivé : le calcul part toujours de la vidéo.
SPIRULA_LOCAL_ROOT = ''  # Installation locale existante désactivée : Spirula vient de SPIRULA_ARCHIVE.
SPIRULA_BINARY_RELATIVE = 'build_vulkan/spirula'
SAM_MODEL_RELATIVE = 'models/sam3-q4_0.ggml'
MAMBA_ENV = 'gsplat'
PHASE = 'all'  # 'prepare' : masques/SfM/découpage ; 'all' : jusqu'au splat.
SAVE_ON_SUCCESS = True
SFM_CAMERA_MODE = 'folder'  # 'image' : une caméra par frame.
SFM_CAMERA_SPLIT = 'false'
SFM_CAMERA_SPLIT_MIN_IMAGES = 100
SFM_FILTER_DENSITY = 'false'  # true : filtrage sparse par densité avant découpage.
SFM_DENSITY_VOTE_FRACTION = '0.6'  # ]0,1[ ; baisser = plus strict.
SFM_DENSITY_FAR_SUPPORT_RATE = '0.35'  # ]0,1] ; monter = plus strict.
SFM_MIN_TRI_ANGLE = '6'  # Angle de triangulation minimal en degrés, ]0,90].

PROJECT = Path('/content/colmap_to_gs')
WORK = Path('/content/colmap_to_gs_work')
if not RUN_NAME or Path(RUN_NAME).name != RUN_NAME or RUN_NAME in ('.', '..'):
    raise ValueError('RUN_NAME doit être un simple nom de dossier.')
if not isinstance(NUM_FRAMES, int) or NUM_FRAMES <= 0 or PHASE not in ('all', 'prepare'):
    raise ValueError('NUM_FRAMES positif et PHASE all/prepare requis.')
if min(TRAIN_NUM_ITER, TRAIN_SP_CAP_MAX, SPLIT_NUM_BLOCKS, NUM_CPUS) <= 0:
    raise ValueError('TRAIN_NUM_ITER, TRAIN_SP_CAP_MAX, SPLIT_NUM_BLOCKS et NUM_CPUS doivent être positifs.')
if SFM_CAMERA_MODE not in ('folder', 'single', 'image'):
    raise ValueError('SFM_CAMERA_MODE doit valoir folder, single ou image.')
if SFM_SP_PAIRS not in ('prefilter', 'sequential', 'auto', 'exhaustive'):
    raise ValueError('SFM_SP_PAIRS doit valoir prefilter, sequential, auto ou exhaustive.')
if SFM_FILTER_DENSITY not in ('true', 'false'):
    raise ValueError('SFM_FILTER_DENSITY doit valoir true ou false.')
if not 0 < float(SFM_MIN_TRI_ANGLE) <= 90:
    raise ValueError('SFM_MIN_TRI_ANGLE doit être dans ]0,90].')
if not (0 < float(SFM_DENSITY_VOTE_FRACTION) < 1 and 0 < float(SFM_DENSITY_FAR_SUPPORT_RATE) <= 1):
    raise ValueError('SFM_DENSITY_VOTE_FRACTION doit être dans ]0,1[ et SFM_DENSITY_FAR_SUPPORT_RATE dans ]0,1].')
if SFM_CAMERA_SPLIT not in ('true', 'false') or SFM_CAMERA_SPLIT_MIN_IMAGES <= 0:
    raise ValueError('SFM_CAMERA_SPLIT true/false et SFM_CAMERA_SPLIT_MIN_IMAGES positif requis.')
if SFM_CAMERA_SPLIT == 'true' and SFM_CAMERA_MODE == 'image':
    raise ValueError('SFM_CAMERA_SPLIT=true est inutile avec SFM_CAMERA_MODE=image.')
if SFM_SP_PAIRS == 'sequential' and SFM_CAMERA_SPLIT == 'true':
    raise ValueError('SFM_SP_PAIRS=sequential exige SFM_CAMERA_SPLIT=false pour préserver l’ordre temporel.')
if PIPELINE_RESUME not in ('true', 'false') or REMOVE_LOCKS not in ('true', 'false'):
    raise ValueError('PIPELINE_RESUME et REMOVE_LOCKS doivent être true ou false.')
if REMOVE_LOCKS == 'true' and PIPELINE_RESUME != 'true':
    raise ValueError('REMOVE_LOCKS=true exige PIPELINE_RESUME=true.')
if not MAMBA_ENV or Path(MAMBA_ENV).name != MAMBA_ENV or MAMBA_ENV in ('.', '..'):
    raise ValueError('MAMBA_ENV doit être un simple nom.')
if MASKS not in ('on', 'off'):
    raise ValueError('MASKS doit valoir on ou off.')
RUN = WORK / RUN_NAME
OUTPUT = RUN / 'output'
FRAMES = RUN / 'frames'
LOGS = RUN / 'logs'
LOGS.mkdir(parents=True, exist_ok=True)
print('Calcul local :', RUN, '\nSauvegardes :', DRIVE_BACKUPS)


## 1. Drive et code
Le dépôt est mis à jour sans écraser de modifications locales. La branche choisie contient le notebook, son helper et l'environnement Conda.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
required_inputs = [Path(COLMAP_ARCHIVE) if COLMAP_ARCHIVE else VIDEO_DRIVE]
if not SPIRULA_LOCAL_ROOT:
    required_inputs.append(SPIRULA_ARCHIVE)
for path in required_inputs:
    if not path.is_file():
        raise FileNotFoundError(path)
if PROJECT.exists():
    if not (PROJECT / '.git').is_dir():
        raise RuntimeError(f'{PROJECT} existe sans dépôt Git ; choisir un autre PROJECT.')
    dirty = subprocess.check_output(['git', '-C', str(PROJECT), 'status', '--porcelain'], text=True)
    if dirty.strip():
        raise RuntimeError('Dépôt modifié : sauvegarder les changements avant mise à jour.\n'+dirty)
    for args in (['fetch', 'origin', GIT_BRANCH+':refs/remotes/origin/'+GIT_BRANCH], ['checkout', GIT_BRANCH], ['pull', '--ff-only', 'origin', GIT_BRANCH]):
        subprocess.run(['git', '-C', str(PROJECT), *args], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', GIT_BRANCH, 'https://github.com/NicoIGN/colmap_to_gs.git', str(PROJECT)], check=True)
sys.path.insert(0, str(PROJECT / 'environment/google.colab'))
import importlib, colab_runtime
importlib.reload(colab_runtime)
from colab_runtime import configure_nvidia_vulkan, ensure_conda_environment, prepare_video_frames, restore_colmap, run_live, restore_spirula, save_colmap_zip, save_results_zip, sha256
def run(command, name, **kwargs):
    return run_live(command, LOGS, name, **kwargs)
run(['git', '-C', str(PROJECT), 'submodule', 'update', '--init', '--recursive'], 'submodules')
run(['git', '-C', str(PROJECT), 'log', '-1', '--oneline'], 'version')


## 2. Dépendances et Vulkan : arrêter ici si NVIDIA n'est pas visible
Pas d'installation automatique de pilote NVIDIA : le pilote doit correspondre à celui du runtime Colab. Les erreurs `DISPLAY`/`XDG_RUNTIME_DIR` peuvent être liées au mode sans écran ; une erreur de création d'instance ou l'absence de GPU NVIDIA est bloquante.


In [ ]:
run(['nvidia-smi'], 'nvidia')
run(['apt-get', 'update', '-qq'], 'apt-update')
run(['apt-get', 'install', '-y', '--no-install-recommends', 'git-lfs', 'curl', 'bzip2', 'libvulkan1', 'vulkan-tools', 'libgl1'], 'apt-install', env={'DEBIAN_FRONTEND': 'noninteractive'})


### Diagnostic NVIDIA / Vulkan (avant tout calcul)
Cette cellule ne modifie ni les pilotes ni les variables d'environnement. Les commandes en échec sont signalées sans interrompre les autres diagnostics ; **cela ne valide pas Vulkan**. La configuration et le contrôle bloquant suivent.
Le rapport complet reste dans les logs locaux. Mettre `DIAGNOSTIC_LD_DEBUG=True` pour tracer aussi les bibliothèques réellement chargées : le rapport peut être volumineux, mais l'affichage est limité. Après un essai de runtime isolé, relancer cette cellule avec les mêmes variables que le calcul. Un ICD Vulkan SC n'est pas un ICD Vulkan standard.


In [ ]:
import json, shutil, tempfile
DIAGNOSTIC_LD_DEBUG = False

with tempfile.NamedTemporaryFile(mode='w', encoding='utf-8', prefix='vulkan-diagnostic-', suffix='.log', dir=LOGS, delete=False) as diagnostic_report:
    def diagnostic_print(text):
        print(text, flush=True)
        diagnostic_report.write(str(text)+'\n')
        diagnostic_report.flush()

    def diagnostic_command(command, *, child_env=None, loader_trace=False):
        diagnostic_print('\n=== '+ ' '.join(command)+' ===')
        if shutil.which(command[0]) is None:
            diagnostic_print('Commande indisponible : '+command[0])
            return ''
        try:
            result = subprocess.run(command, env=child_env, text=True, errors='replace', stdout=subprocess.PIPE, stderr=subprocess.STDOUT, timeout=60)
        except subprocess.TimeoutExpired as error:
            partial = error.stdout or b''
            diagnostic_print(partial.decode(errors='replace') if isinstance(partial, bytes) else partial)
            diagnostic_print('Délai de 60 s dépassé ; diagnostic incomplet.')
            return ''
        diagnostic_report.write(result.stdout)
        diagnostic_report.flush()
        lines = result.stdout.splitlines()
        if loader_trace:
            lines = [line for line in lines if any(word in line.lower() for word in ('error', 'warning', 'undefined symbol', 'not found', 'devicename', 'devicetype'))]
        print('\n'.join(lines[:120]), flush=True)
        if len(lines) > 120:
            print('Affichage limité à 120 lignes ; sortie complète dans le rapport.', flush=True)
        diagnostic_print('Code de sortie : '+str(result.returncode))
        return result.stdout

    diagnostic_command(['nvidia-smi', '--query-gpu=name,driver_version', '--format=csv,noheader'])
    diagnostic_command(['dpkg-query', '-W', '-f=${Package} ${Version}\n', 'libnvidia-*'])
    diagnostic_print('\n=== Variables de sélection et de chargement ===')
    for key in sorted(os.environ):
        if key.startswith('VK_') or key in ('__EGL_VENDOR_LIBRARY_FILENAMES', 'LD_LIBRARY_PATH', 'LD_PRELOAD', 'LIBGL_ALWAYS_SOFTWARE', 'NVIDIA_DRIVER_CAPABILITIES', 'NVIDIA_VISIBLE_DEVICES'):
            diagnostic_print(key+'='+os.environ[key])
    cache = diagnostic_command(['ldconfig', '-p'])
    diagnostic_print('\n=== Bibliothèques NVIDIA et dépendances GLX/EGL ===')
    libraries = {Path(line.split(' => ', 1)[1]) for line in cache.splitlines() if ' => ' in line and 'nvidia' in line.lower()}
    for directory in os.environ.get('LD_LIBRARY_PATH', '').split(':')+['/usr/lib64-nvidia', '/usr/local/nvidia/lib64', '/usr/lib/x86_64-linux-gnu']:
        if directory:
            libraries.update(Path(directory).glob('lib*nvidia*.so*'))
    if not libraries:
        diagnostic_print('Aucune bibliothèque NVIDIA trouvée dans les chemins inspectés.')
    for library in sorted(libraries):
        diagnostic_print(str(library)+' -> '+str(library.resolve()))
        if library.name.startswith(('libGLX_nvidia.so', 'libEGL_nvidia.so')):
            diagnostic_command(['ldd', str(library)])
    diagnostic_print('\n=== Descripteurs Vulkan ===')
    icds = set()
    for directory in ('/usr/share/vulkan/icd.d', '/etc/vulkan/icd.d', '/usr/local/share/vulkan/icd.d'):
        icds.update(Path(directory).glob('*.json'))
    for key in ('VK_DRIVER_FILES', 'VK_ICD_FILENAMES', 'VK_ADD_DRIVER_FILES'):
        icds.update(Path(path) for path in os.environ.get(key, '').split(':') if path)
    if not icds:
        diagnostic_print('Aucun descripteur Vulkan trouvé dans les chemins inspectés.')
    for icd in sorted(icds):
        diagnostic_print(str(icd))
        try:
            diagnostic_print(json.dumps(json.loads(icd.read_text()), indent=2))
        except (OSError, json.JSONDecodeError) as error:
            diagnostic_print('ICD illisible ou invalide : '+str(error))
    loader_env = dict(os.environ, VK_LOADER_DEBUG='error,warn,driver')
    if DIAGNOSTIC_LD_DEBUG:
        loader_env['LD_DEBUG'] = 'libs'
    diagnostic_command(['vulkaninfo', '--summary'], child_env=loader_env, loader_trace=DIAGNOSTIC_LD_DEBUG)
    diagnostic_print('\nRapport local : '+diagnostic_report.name)


### Configuration Vulkan et EGL NVIDIA pour Colab
Les bibliothèques NVIDIA sont déjà montées dans `/usr/lib64-nvidia`, mais leurs descripteurs peuvent manquer. Cette cellule crée les descripteurs Vulkan **et EGL** dans le workspace local, sans installer de pilote ni modifier CUDA. Les variables sont transmises aux processus suivants seulement après un test Vulkan réussi. Relancer cette cellule après chaque redémarrage du runtime.


In [ ]:
configure_nvidia_vulkan(Path('/usr/lib64-nvidia'), WORK, LOGS)


### Contrôle Vulkan bloquant
Ne pas poursuivre si cette cellule échoue, même si la cellule de diagnostic a terminé.


In [ ]:
for directory in (Path('/usr/share/vulkan/icd.d'), Path('/etc/vulkan/icd.d')):
    print('ICD :', directory, [p.name for p in directory.glob('*.json')], flush=True)
print('Variables Vulkan :', {k: v for k, v in os.environ.items() if k.startswith('VK_')}, flush=True)
vulkan_log = run(['vulkaninfo', '--summary'], 'vulkan-system')
vulkan_text = vulkan_log.read_text(errors='replace')
if 'NVIDIA' not in vulkan_text or 'deviceType' not in vulkan_text:
    raise RuntimeError('Vulkan ne voit pas NVIDIA. Ne pas lancer Spirula. Examiner le log et les ICD du runtime.')


## 3. Environnement de calcul (long au premier lancement)
Un environnement identique est réutilisé. Un environnement existant non marqué ou un YAML modifié déclenche une synchronisation explicite, sans effacer l'environnement. Les contrôles CUDA et Vulkan sont réexécutés.


In [ ]:
import shutil, tarfile, urllib.request, tempfile
MAMBA = Path('/content/bin/micromamba')
MAMBA_ROOT = Path('/content/micromamba')
os.environ['MAMBA_ROOT_PREFIX'] = str(MAMBA_ROOT)
if not MAMBA.is_file():
    MAMBA.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory(dir=WORK) as temp:
        package = Path(temp) / 'micromamba.tar.bz2'
        print('Téléchargement micromamba...', flush=True)
        urllib.request.urlretrieve('https://micro.mamba.pm/api/micromamba/linux-64/latest', package)
        with tarfile.open(package) as tar:
            member = tar.getmember('bin/micromamba')
            if not member.isfile():
                raise RuntimeError('Archive micromamba invalide')
            with tar.extractfile(member) as source, MAMBA.open('wb') as target:
                shutil.copyfileobj(source, target)
    MAMBA.chmod(0o755)
ENV_FILE = PROJECT / 'environment/google.colab/conda_env.yml'
ENV_PREFIX = MAMBA_ROOT / 'envs' / MAMBA_ENV
ensure_conda_environment(MAMBA, MAMBA_ENV, ENV_PREFIX, ENV_FILE, LOGS)
def in_env(command):
    return [str(MAMBA), 'run', '-n', MAMBA_ENV, *map(str, command)]
run(in_env(['python', '-u', '-c', "import torch, cv2, numpy, PIL, tqdm; print('CUDA:', torch.cuda.is_available(), flush=True); assert torch.cuda.is_available(), 'CUDA indisponible'; print(torch.cuda.get_device_name(0), flush=True)"]), 'python-gpu')
env_vulkan = run(in_env(['vulkaninfo', '--summary']), 'vulkan-conda')
if 'NVIDIA' not in env_vulkan.read_text(errors='replace'):
    raise RuntimeError('Vulkan NVIDIA absent dans Conda : vérifier ses bibliothèques et variables VK_* avant le calcul.')


## 4. Spirula et données locales
Spirula est réutilisé par SHA256 ou depuis `SPIRULA_LOCAL_ROOT`. Sans ZIP COLMAP, la vidéo est copiée et vérifiée localement. Avec `COLMAP_ARCHIVE`, le ZIP Drive est copié, extrait et validé ; vidéo et SfM sont sautés. Un import existant est vérifié puis réutilisé sans écraser un autre workspace.


In [ ]:
SPIRULA_ROOT, SPIRULA_BIN = restore_spirula(SPIRULA_ARCHIVE, WORK / 'assets', SPIRULA_BINARY_RELATIVE, existing_root=SPIRULA_LOCAL_ROOT)
SAM_MODEL = SPIRULA_ROOT / SAM_MODEL_RELATIVE
if not SAM_MODEL.is_file():
    raise FileNotFoundError(f'Checkpoint SAM absent : {SAM_MODEL}')
libraries = run(in_env(['ldd', str(SPIRULA_BIN)]), 'spirula-libraries')
if 'not found' in libraries.read_text(errors='replace'):
    raise RuntimeError('Bibliothèques Spirula absentes : voir le log ldd ; archive/runtime incompatibles.')
VIDEO_LOCAL = None
if COLMAP_ARCHIVE:
    validator = in_env(['python', PROJECT / 'tools/images_to_splat/resume.py'])
    SOURCE_IMAGES = restore_colmap(Path(COLMAP_ARCHIVE), RUN, validator, LOGS)
else:
    if (RUN / '.colmap-import.json').exists():
        raise RuntimeError('Ce run provient d’un ZIP COLMAP : conserver COLMAP_ARCHIVE ou choisir un nouveau RUN_NAME.')
    SOURCE_IMAGES = FRAMES
    VIDEO_LOCAL = RUN / ('input'+VIDEO_DRIVE.suffix)
    video_hash = sha256(VIDEO_DRIVE)
    if VIDEO_LOCAL.exists():
        if sha256(VIDEO_LOCAL) != video_hash:
            raise RuntimeError('Vidéo locale différente/incomplète : choisir un nouveau RUN_NAME.')
        print('Vidéo locale inchangée : réutilisée.', flush=True)
    else:
        print('Copie vidéo Drive -> disque local...', flush=True)
        temporary_video = VIDEO_LOCAL.with_suffix(VIDEO_LOCAL.suffix+'.partial')
        shutil.copyfile(VIDEO_DRIVE, temporary_video)
        if sha256(temporary_video) != video_hash:
            raise IOError('Copie vidéo non conforme ; fichier .partial conservé pour diagnostic.')
        temporary_video.rename(VIDEO_LOCAL)
print('Spirula :', SPIRULA_BIN, '\nSAM :', SAM_MODEL, '\nSource images :', SOURCE_IMAGES)


## 5. Réglages
OUTPUT_DIR est strictement local. PIPELINE_RESUME ne permet pas de changer images/modèle/paramètres dans un même workspace. Les frames sont extraites par le script vidéo, jamais séparément ici.


In [ ]:
import shlex
if ACCEPT_SAM_LICENSE is not True:
    raise RuntimeError('Lire la licence SAM puis définir ACCEPT_SAM_LICENSE=True si vous l’acceptez.')
SETTINGS = {
    'PIPELINE_RESUME': PIPELINE_RESUME, 'SPIRULA_ROOT': str(SPIRULA_ROOT), 'SPIRULA_BIN': str(SPIRULA_BIN),
    'REMOVE_LOCKS': REMOVE_LOCKS,
    'PYTHON_BIN': 'python3', 'SFM_IMAGES_DIR': str(SOURCE_IMAGES), 'SFM_IMAGES_RECURSIVE': 'true',
    'OUTPUT_DIR': str(OUTPUT), 'COLMAP_METHOD': 'spirula', 'TRAIN_BACKEND': 'spirula',
    'SFM_MASKS': MASKS, 'TRAIN_MASKS': MASKS, 'TRAIN_MASKS_BACKEND': 'spirula',
    'TRAIN_MASKS_SPIRULA_MODEL': str(SAM_MODEL), 'TRAIN_MASKS_LABELS': 'sky,clouds',
    'TRAIN_MASKS_MODE': 'empty', 'SPIRULA_DEVICE': 'auto',
    'NUM_CPUS': str(NUM_CPUS), 'MASKS_ACCEPT_LICENSE': 'true',
    'SFM_MAX_IMAGE_SIZE': '2048', 'SFM_ALIKED_MAX_FEATURES': '4096',
    'SFM_SP_QUALITY': 'medium', 'SFM_SP_PAIRS': SFM_SP_PAIRS, 'SFM_CAMERA_MODE': SFM_CAMERA_MODE, 'SFM_MIN_TRI_ANGLE': SFM_MIN_TRI_ANGLE,
    'SFM_FILTER_DENSITY': SFM_FILTER_DENSITY, 'SFM_DENSITY_VOTE_FRACTION': SFM_DENSITY_VOTE_FRACTION, 'SFM_DENSITY_FAR_SUPPORT_RATE': SFM_DENSITY_FAR_SUPPORT_RATE,
    'SFM_CAMERA_SPLIT': SFM_CAMERA_SPLIT, 'SFM_CAMERA_SPLIT_MIN_IMAGES': str(SFM_CAMERA_SPLIT_MIN_IMAGES),
    'SPLIT_NUM_BLOCKS': str(SPLIT_NUM_BLOCKS), 'SPLIT_OVERLAP_MAX_FRACTION': '0.10',
    'TRAIN_NUM_ITER': str(TRAIN_NUM_ITER), 'TRAIN_SP_PRESET': '3dgs', 'TRAIN_SP_QUALITY': 'high',
    'TRAIN_SP_CAP_MAX': str(TRAIN_SP_CAP_MAX), 'TRAIN_SP_REFINE_STOP_ITER': str(min(10000, TRAIN_NUM_ITER)),
    'TRAIN_SP_REFINE_STOP_NUM_ITER': str(min(2000, TRAIN_NUM_ITER - 1)), 'TRAIN_SP_FLOATER_SUPPRESSION': 'mild', 'TRAIN_CLEAN': 'false',
    'EXPORT_SPLAT_NAME': RUN_NAME+'.splat',
}
CONFIG = RUN / 'colab_video_config.sh'
CONFIG.write_text('#!/usr/bin/env bash\nset -Eeuo pipefail\n'+''.join(f'{key}={shlex.quote(value)}\n' for key, value in SETTINGS.items())+'export PYTHONUNBUFFERED=1\nexport SS_NN_LOG=1\n')
print('Configuration :', CONFIG, '\nPhase :', PHASE, '\nRésultats locaux :', OUTPUT, flush=True)


## 6. Lancer : logs immédiats
L'affichage gère aussi les barres de progression sans saut de ligne. Si un programme natif tamponne lui-même sa sortie, le helper ne peut pas le forcer à émettre plus tôt. Interrompre la cellule arrête le groupe de processus ; les logs ne sont pas écrasés à la relance.


In [ ]:
if not COLMAP_ARCHIVE:
    FRAMES.mkdir(parents=True, exist_ok=True)
    run(in_env(['bash', PROJECT / 'scripts/images_to_3dgs.sh', CONFIG, '--validate-only']), 'preflight', cwd=PROJECT)
    extractor = in_env(['python', PROJECT / 'tools/preprocess/extract_sharp_frames.py'])
    prepare_video_frames(VIDEO_LOCAL, FRAMES, NUM_FRAMES, extractor, LOGS)
colmap_zip = result_zip = None
print('Calcul local ; seuls les ZIP COLMAP et 3DGS seront envoyés sur Drive.', flush=True)
run(in_env(['bash', 'scripts/images_to_3dgs.sh', str(CONFIG), '--phase', 'prepare']), 'pipeline-prepare', cwd=PROJECT)
if SAVE_ON_SUCCESS:
    colmap_zip = save_colmap_zip(OUTPUT, DRIVE_BACKUPS)
if PHASE == 'all':
    command = in_env(['bash', 'scripts/images_to_3dgs.sh', str(CONFIG), '--phase', 'all'])
    run(command, 'pipeline-all', cwd=PROJECT)
    if SAVE_ON_SUCCESS:
        result_zip = save_results_zip(OUTPUT, DRIVE_BACKUPS, SETTINGS['EXPORT_SPLAT_NAME'])
print('Phase terminée :', PHASE, flush=True)
print('Les artefacts intermédiaires restent uniquement dans :', RUN, flush=True)


## 7. Export du résultat final uniquement (ZIP)
Après l'entraînement, le ZIP contient uniquement le `.splat` final, son `.georef.json` si disponible et les métadonnées géographiques référencées (résumé et GPX source). Aucun checkpoint, masque, log, vidéo ou frame. La copie Drive est vérifiée par SHA256. Cette cellule réutilise l'export automatique déjà créé ; sinon elle permet l'export manuel après succès. En phase `prepare`, aucun résultat 3DGS n'est exporté.


In [ ]:
if PHASE == 'all':
    if not globals().get('result_zip') or not result_zip.is_file():
        result_zip = save_results_zip(OUTPUT, DRIVE_BACKUPS, SETTINGS['EXPORT_SPLAT_NAME'])
    print('Résultat final sur Drive :', result_zip)
else:
    print('Phase prepare : seul le ZIP COLMAP est exporté.')


### Export COLMAP seul vers Drive (ZIP)
Le ZIP est exporté automatiquement après préparation, avant l'entraînement, si `SAVE_ON_SUCCESS=True`. Cette cellule réutilise cet export ou permet de le créer manuellement, calcul arrêté. Il contient le modèle global `OUTPUT/sfm/sparse/0` sous `sparse/0/`, et les images `OUTPUT/images` sous `images/` ; les liens sont matérialisés. Aucun checkpoint, masque ou log. Copie Drive horodatée et vérifiée par SHA256.


In [ ]:
if not globals().get('colmap_zip') or not colmap_zip.is_file():
    colmap_zip = save_colmap_zip(OUTPUT, DRIVE_BACKUPS)
print('Archive COLMAP sur Drive :', colmap_zip)


## 8. Résultats et diagnostic (sans lister toutes les images)


In [ ]:
import json
splats = sorted(OUTPUT.rglob('*.splat')) if OUTPUT.exists() else []
for path in splats:
    print(f'{path.stat().st_size/2**20:.1f} Mio  {path}')
if not splats:
    print('Pas encore de splat exporté ; consulter les logs locaux :', LOGS)
report = OUTPUT / 'pipeline_report.json'
if report.is_file():
    for entry in json.loads(report.read_text()).get('runs', [])[-5:]:
        print(entry.get('pipeline'), entry.get('status'), entry.get('duration_seconds'))
